In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("2")

2


In [2]:
# 1. Load the dataset
dataset_path = 'Updated_Imputed_Dataset.csv'
data = pd.read_csv(dataset_path)

C:\Users\dinit\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3457: DtypeWarning: Columns (17,25,26,27,29,30,32,34,35,36,37,38,41,43,44,45,46,47,48,49,50,51,52) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:

# 2. Define the columns and preprocess data
selected_columns_updated = [
    'Perpetrator Sex', 'Perpetrator Age', 'Perpetrator Race', 'Perpetrator Ethnicity', 'Relationship',
    'Victim Count', 'Offender Name', 'Offender Appearance', 'Offender Job', 'Offender Age', 'Offender Height',
    'Offender Weight', 'Offender Personality', 'Offender Education and Family Background', 'Offender Motive',
    'Offender Planned or Unplanned', 'Offender Regular or One-Time', 'Offender No. of Homicides',
    'Offender Part of Organization', 'Victim If Randomly Chosen or Planned'
]

# Subset the data
subset_data_updated = data[selected_columns_updated]

# Handle missing values by filling with the median for numerical columns
subset_data_filled_updated = subset_data_updated.fillna(subset_data_updated.median(numeric_only=True))

# For categorical columns, apply label encoding
categorical_columns_updated = subset_data_filled_updated.select_dtypes(include=['object']).columns
label_encoders_updated = {}

for col in categorical_columns_updated:
    le = LabelEncoder()
    subset_data_filled_updated[col] = le.fit_transform(subset_data_filled_updated[col].astype(str))
    label_encoders_updated[col] = le  # Save label encoder for possible inverse transformation

# Now scale the selected data (numeric columns only)
numeric_data_updated = subset_data_filled_updated.select_dtypes(include=['float64', 'int64'])

# Normalize the numerical data
scaler = StandardScaler()
numeric_data_scaled_updated = scaler.fit_transform(numeric_data_updated)

# 3. Define the symbolic rules (Knowledge Representation) and the neural network
class ExtendedPenaltyLogic:
    def __init__(self):
        # Define symbolic rules for demographics, psychological traits, etc.
        self.rules = {
            'male': 1,
            'white': 1,
            'age_20_40': 1,
            'childhood_abuse': 1,
            'psychopathy': 1,
            'high_intelligence': 1,
            'sexual_deviance': 1,
            'compulsive_behavior': 1,
            'lack_of_empathy': 1,
            'narcissistic_traits': 1,
            'chronic_boredom': 1,
            'sadistic_tendencies': 1,
            'fantasy_driven_motives': 1,
            'revenge_driven_motives': 1,
            'thrill_killing': 1,
            'predatory_nature': 1,
            'victim_stalking': 1,
            'escalating_violence': 1,
            'repetitive_criminal_behavior': 1,
            'organized_scene': 1,
            'disorganized_scene': 1,
            'trophy_taking': 1,
            'signature_crime_pattern': 1,
            'use_of_ritual': 1,
            'gory_crime_scene': 1,
            'vulnerable_victims': 1,
            'specific_victim_type': 1,
            'random_victims': 1,
            'sexual_violence': 1,
            'charming': 1,
            'manipulative': 1,
            'isolated': 1,
            'egocentric': 1,
            'lack_of_remorse': 1,
            'minimal_emotional_expression': 1,
            'enjoyment_of_suffering': 1,
            'comfort_with_violence': 1,
            'manipulating_authority': 1,
            'low_profile': 1,
            'obsession_with_planning': 1,
            'avoid_confrontation': 1
        }

    def apply_rules(self, inputs):
        # Here we will use the input data to check if any symbolic rule applies and return its outcome
        results = {}
        # Example: If a victim is male, apply the rule that male serial killers are predominant
        results['male'] = self.rules['male'] if inputs['victim_sex'] == 1 else 0  # 1 for male, 0 for female
        # Add more logic for the other rules as needed (this part can be expanded)

        return results

# 4. Define the Neural Network with KR rules integrated into the loss function
class NeuralNetworkWithKR:
    def __init__(self, num_visible, num_hidden):
        self.num_visible = num_visible
        self.num_hidden = num_hidden
        self.weights = np.random.randn(num_visible, num_hidden)  # Initialize weights randomly
        self.hidden_bias = np.zeros(num_hidden)
        self.visible_bias = np.zeros(num_visible)
        self.kr = ExtendedPenaltyLogic()  # Include extended KR inside the neural network
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def sample_hidden(self, visible):
        activation = np.dot(visible, self.weights) + self.hidden_bias
        return self.sigmoid(activation)
    
    def sample_visible(self, hidden):
        activation = np.dot(hidden, self.weights.T) + self.visible_bias
        return self.sigmoid(activation)
    
    def apply_kr_penalty(self, inputs):
        kr_results = self.kr.apply_rules(inputs)
        
        # Apply penalties based on symbolic rule violations
        penalty = 0
        for condition, value in kr_results.items():
            if value == True:  # If the rule predicts "True" (e.g., male, psychopathy) and NN contradicts it
                prediction = self.run(inputs)
                if prediction[0] < 0.5:  # Violation
                    penalty = 1  # Increase penalty
            elif value == False:  # If the rule predicts "False" and NN contradicts it
                prediction = self.run(inputs)
                if prediction[0] > 0.5:  # Violation
                    penalty = 1  # Increase penalty
        return penalty
    
    def run(self, visible):
        hidden = self.sample_hidden(visible)
        return hidden
    
    def train(self, data, learning_rate=0.1, epochs=1000):
        for epoch in range(epochs):
            for visible in data:
                hidden = self.sample_hidden(visible)
                visible_reconstructed = self.sample_visible(hidden)
                hidden_reconstructed = self.sample_hidden(visible_reconstructed)

                # Apply the KR penalty during training
                penalty = self.apply_kr_penalty(visible)
                if penalty > 0:
                    # Adjust weights based on KR violation penalty
                    self.weights -= learning_rate * penalty  # Apply penalty to weights

                # Standard Contrastive Divergence update
                self.weights += learning_rate * (np.outer(visible, hidden) - np.outer(visible_reconstructed, hidden_reconstructed))
                self.visible_bias += learning_rate * (visible - visible_reconstructed)
                self.hidden_bias += learning_rate * (hidden - hidden_reconstructed)

            if epoch % 50 == 0:
                print(f"Epoch {epoch} completed")

    def test(self, inputs):
        hidden = self.run(inputs)
        return hidden


# 5. Train the model using the preprocessed dataset
model = NeuralNetworkWithKR(num_visible=numeric_data_scaled_updated.shape[1], num_hidden=6)

# Train the model (simplified target value for demonstration purposes)
target = np.array([1])  # Target is simplified to 1 for the demo

# Train the model (batch_size is 1 for simplicity)
model.train(numeric_data_scaled_updated, epochs=10)

# 6. Testing with new data
test_input = np.array([28, 80, 75])  # Example input for testing: temperature, humidity, cloudiness
test_input_scaled = scaler.transform(test_input.reshape(1, -1))  # Scaling test input

# Predict the output (0 or 1)
prediction = model.test(test_input_scaled)
print(f"Prediction for test input: {prediction[0]}")


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices